# PartQC — treino do modelo no Colab (SPEC-012)

Notebook **fino**: só chama o pacote `qc_ml`. Toda a lógica (treino, export ONNX, limiares na validação, avaliação no teste) está em `ml/src/qc_ml/` e é testada no repositório.

Antes de rodar:

1. **Ambiente de execução → Alterar o tipo → GPU** (T4 gratuita).
2. **Segredo do Colab** (ícone de chave, à esquerda): crie `GITHUB_RO_TOKEN` com um token *fine-grained* do GitHub **somente leitura** (permissão *Contents: Read-only*, só neste repositório) e dê acesso a este notebook. O token nunca é escrito no notebook (R-03).
3. No Google Drive pessoal, em `MyDrive/partqc/datasets/`, o `.zip` do dataset gerado pela dupla (`docs/dataset-protocol.md` §8, com a checagem de privacidade feita antes do upload).

Os dados chegam ao Colab pelo Drive pessoal (ADR-004); nada sobe para a nuvem do projeto (R-02). **Não salve este notebook com saídas** no repositório.

In [ ]:
import sys

import torch

assert sys.version_info[:2] == (3, 12), (
    f"PartQC needs Python 3.12 (R-12); this runtime has {sys.version.split()[0]}"
)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU"
print("Python", sys.version.split()[0], "| GPU:", torch.cuda.get_device_name(0))

## 1. Google Drive

Ajuste `DATASET_ZIP` para o arquivo do dataset. Os checkpoints ficam no Drive, para que `--resume` funcione depois de uma desconexão.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/partqc"
DATASET_ZIP = f"{DRIVE_DIR}/datasets/partqc-data-AAAAMMDD.zip"
MODEL_VERSION = "partqc-mnv3-v1"
CONFIG = f"ml/configs/{MODEL_VERSION}.json"

## 2. Código do repositório

Clona a `main` com o token somente leitura, passado ao git só pelo ambiente do processo (não fica na URL, no `.git/config` nem na saída). Alternativa sem token: descompactar um snapshot do repositório salvo no Drive, com `!unzip -q "$DRIVE_DIR/repo-snapshot.zip" -d /content/partqc`.

In [ ]:
import base64
import os
import subprocess

from google.colab import userdata

REPO_URL = "https://github.com/CarlosZeyy/Projeto_IoT.git"
REPO_DIR = "/content/partqc"
basic = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_RO_TOKEN')}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.extraHeader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}",
}
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], env=git_env, check=True)
del basic, git_env
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Instalação

Pacotes do repositório em modo editável, com o extra `train` (torch, timm, onnx, coral-pytorch). As versões instaladas ficam registradas no `training_config.json`.

In [ ]:
!pip install -q -e packages/contracts -e packages/preprocess -e "ml[train]"
assert _exit_code == 0, "pip install failed"  # noqa: F821 (_exit_code: set by IPython after !)

## 4. Dataset

Descompacta o dataset em `/content/data` e liga `ml/data` a ele. Os checkpoints vão para o Drive.

In [ ]:
!mkdir -p /content/data && unzip -q -o "$DATASET_ZIP" -d /content/data
assert _exit_code == 0, "unzip failed"  # noqa: F821 (_exit_code: set by IPython after !)

In [ ]:
from pathlib import Path

manifests = sorted(Path("/content/data").rglob("manifest.csv"))
assert len(manifests) == 1, f"expected one manifest.csv in the zip, found {manifests}"
for link, target in (
    (Path("ml/data"), manifests[0].parent),
    (Path("ml/checkpoints"), Path(DRIVE_DIR) / "checkpoints"),
):
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    assert not link.exists(), f"{link} exists and is not a link"
    link.symlink_to(target, target_is_directory=True)
print("dataset:", manifests[0].parent)

## 5. Validação, treino, export, limiares e avaliação

O split versionado (`ml/splits/split_v1.csv`) vem do repositório. O pipeline para no primeiro erro: os limiares saem da **validação** e as métricas do **teste** (R-11). A latência medida aqui é só referência; a oficial (RNF-02) é medida no notebook da borda com `python -m qc_ml.evaluate --latency-only`.

In [ ]:
!python -m qc_ml.data validate --dataset-dir ml/data
assert _exit_code == 0, "the dataset has problems: fix them before training"  # noqa: F821 (_exit_code: set by IPython after !)

In [ ]:
!python -m qc_ml.pipeline --config $CONFIG --latency-machine "Colab CPU (reference only)"
assert _exit_code == 0, "pipeline failed: read the log above"  # noqa: F821 (_exit_code: set by IPython after !)

Se o Colab desconectar no meio do treino, rode de novo as células 1 a 4 e continue do último checkpoint com a célula abaixo (tire o `#`).

In [ ]:
# !python -m qc_ml.pipeline --config $CONFIG --resume --latency-machine "Colab CPU (reference only)"

## 6. Artefatos para o Drive

Copia `ml/models/<versão>/` (com o `model.onnx`) e `ml/reports/<versão>/` para o Drive. Depois, no repositório local, siga o CA-28 da SPEC-012: copiar os artefatos, `evaluate --latency-only` no notebook da borda, gerar a entrada do `ml/models/CHANGELOG.md`, rodar a paridade com o `model.onnx` real e publicar o Release (`ml/README.md`).

In [ ]:
!mkdir -p "$DRIVE_DIR/models" "$DRIVE_DIR/reports"
!cp -r "ml/models/$MODEL_VERSION" "$DRIVE_DIR/models/"
assert _exit_code == 0, "copy to Drive failed"  # noqa: F821 (_exit_code: set by IPython after !)
!cp -r "ml/reports/$MODEL_VERSION" "$DRIVE_DIR/reports/"
assert _exit_code == 0, "copy to Drive failed"  # noqa: F821 (_exit_code: set by IPython after !)
!cat "ml/models/$MODEL_VERSION/model.sha256"